# 16.1 American Put Inverse PINN

This notebook keeps the settings from Notebook 16 and repeats the inverse PINN experiment 10 times.

The neural network initialization, collocation points, terminal points, and boundary time points are newly randomized for every run.

Two price MSEs are recorded for every run:

\[
\mathrm{MSE}_1
=
rac{1}{N}\sum_i
\left(V_{\mathrm{PINN},i}-V_{\mathrm{true},i}ight)^2
\]

\[
\mathrm{MSE}_2
=
rac{1}{N}\sum_i
\left(V_{\mathrm{CRR},i}(\hat r,\hat\sigma)-V_{\mathrm{true},i}ight)^2
\]

The final table reports the average and standard deviation across the 10 runs.

## 1. Imports and configuration

In [ ]:
import os
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as Fnn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

K = 100.0
T = 1.0
r_true = 0.05
sigma_true = 0.20
q = 0.0

r_initial = 0.10
sigma_initial = 0.40

S_min = 0.0
S_max = 200.0

AMERICAN_DATA_CSV = "15_american_synthetic_data.csv"

continuation_data_eps_u = 1e-4
min_informative_price_u = 1e-4
continuation_data_weight = 1.0

N_f = 2000
N_terminal = 500
N_boundary_each = 500

hidden_width = 64
hidden_layers = 4

epochs = 80000
lr_nn = 1e-3
lr_r = 1e-3
lr_sigma = 1e-3
print_every = 500

lambda_data = 1e3
lambda_vi = 1.0
lambda_tv = 1.0
lambda_terminal = 1.0
lambda_boundary_left = 1.0
lambda_boundary_right = 1.0

lambda_bs = 0.0
lambda_eq = 0.0
lambda_pde_cont = 0.0

RUN_LBFGS = False
lbfgs_max_iter = 2000

USE_RIGHT_BOUNDARY = True
continuation_eps_u = 1e-4

N_RUNS = 10

CRR_STEPS = 4000

OUTPUT_DIR = "16_1_american_inverse_10run_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("device:", device)
print("runs:", N_RUNS)
print("true r =", r_true, "true sigma =", sigma_true)
print("initial r =", r_initial, "initial sigma =", sigma_initial)
print("CRR steps for MSE2 =", CRR_STEPS)

## 2. Load the American synthetic observations

In [ ]:
american_df = pd.read_csv(AMERICAN_DATA_CSV)

required_cols = {"t", "S", "V", "u"}
missing = required_cols - set(american_df.columns)
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

t_data_np = american_df["t"].to_numpy(dtype=np.float64)
S_data_np = american_df["S"].to_numpy(dtype=np.float64)
V_data_np = american_df["V"].to_numpy(dtype=np.float64)
u_data_np = american_df["u"].to_numpy(dtype=np.float64)

payoff_data_np = np.maximum(K - S_data_np, 0.0)
payoff_u_data_np = payoff_data_np / K
premium_u_np = u_data_np - payoff_u_data_np

continuation_like_np = premium_u_np > continuation_data_eps_u
exercise_like_np = (S_data_np < K) & (~continuation_like_np)
otm_np = S_data_np >= K

informative_continuation_np = (
    continuation_like_np
    & (u_data_np > min_informative_price_u)
)

continuation_fraction = continuation_like_np.mean()
exercise_like_fraction = exercise_like_np.mean()
otm_fraction = otm_np.mean()
informative_continuation_fraction = informative_continuation_np.mean()

coverage_df = pd.DataFrame([{
    "N_total": len(american_df),
    "continuation_fraction": continuation_fraction,
    "exercise_like_fraction": exercise_like_fraction,
    "OTM_fraction": otm_fraction,
    "informative_continuation_fraction": informative_continuation_fraction,
}])

print("Loaded:", AMERICAN_DATA_CSV)
print("shape:", american_df.shape)
display(coverage_df)
display(american_df.head())

## 3. Fixed observed data tensors

In [ ]:
t_data = torch.tensor(
    t_data_np[:, None],
    device=device,
    dtype=dtype
)

S_data = torch.tensor(
    S_data_np[:, None],
    device=device,
    dtype=dtype
)

u_data = torch.tensor(
    u_data_np[:, None],
    device=device,
    dtype=dtype
)

informative_continuation = torch.tensor(
    informative_continuation_np[:, None],
    device=device,
    dtype=torch.bool
)

data_weights = torch.ones_like(u_data)

if continuation_data_weight != 1.0:
    data_weights = torch.where(
        informative_continuation,
        torch.full_like(u_data, continuation_data_weight),
        data_weights
    )

## 4. Inverse PINN model

In [ ]:
def inverse_softplus(x):
    x = float(x)
    if x <= 0.0:
        raise ValueError("inverse_softplus requires x > 0.")
    return math.log(math.expm1(x))


class AmericanInversePINN(nn.Module):
    def __init__(self, r_initial, sigma_initial):
        super().__init__()

        layers = []
        in_dim = 2

        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width

        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

        self.raw_r = nn.Parameter(
            torch.tensor(
                inverse_softplus(r_initial),
                device=device,
                dtype=dtype
            )
        )

        self.raw_sigma = nn.Parameter(
            torch.tensor(
                inverse_softplus(sigma_initial),
                device=device,
                dtype=dtype
            )
        )

    def normalize_inputs(self, t, S):
        t_n = 2.0 * t / T - 1.0
        S_n = 2.0 * (S - S_min) / (S_max - S_min) - 1.0
        return torch.cat([t_n, S_n], dim=1)

    def r(self):
        return Fnn.softplus(self.raw_r) + 1e-10

    def sigma(self):
        return Fnn.softplus(self.raw_sigma) + 1e-10

    def forward(self, t, S):
        x = self.normalize_inputs(t, S)
        u = Fnn.softplus(self.net(x))
        return u

## 5. American Black Scholes operator

In [ ]:
def american_bs_operator(model, t, S, r, sigma, q=0.0, create_graph=True):
    if not t.requires_grad:
        t = t.clone().detach().requires_grad_(True)

    if not S.requires_grad:
        S = S.clone().detach().requires_grad_(True)

    u = model(t, S)

    u_t = torch.autograd.grad(
        u,
        t,
        grad_outputs=torch.ones_like(u),
        create_graph=create_graph,
        retain_graph=True
    )[0]

    u_S = torch.autograd.grad(
        u,
        S,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    u_SS = torch.autograd.grad(
        u_S,
        S,
        grad_outputs=torch.ones_like(u_S),
        create_graph=create_graph,
        retain_graph=True
    )[0]

    F = (
        u_t
        + 0.5 * sigma**2 * S**2 * u_SS
        + (r - q) * S * u_S
        - r * u
    )

    return F, u, u_t, u_S, u_SS

## 6. Random points for each run

In [ ]:
def rand_uniform(n, low, high):
    return low + (high - low) * torch.rand(
        (n, 1),
        device=device,
        dtype=dtype
    )


def sample_run_points():
    t_f = rand_uniform(N_f, 0.0, T).requires_grad_(True)
    S_f = rand_uniform(N_f, S_min, S_max).requires_grad_(True)

    t_T = torch.full(
        (N_terminal, 1),
        T,
        device=device,
        dtype=dtype
    )

    S_T = rand_uniform(
        N_terminal,
        S_min,
        S_max
    )

    u_T_target = torch.relu(
        torch.tensor(K, device=device, dtype=dtype) - S_T
    ) / K

    t_b0 = rand_uniform(
        N_boundary_each,
        0.0,
        T
    )

    S_b0 = torch.zeros(
        (N_boundary_each, 1),
        device=device,
        dtype=dtype
    )

    u_b0_target = torch.ones_like(t_b0)

    t_bmax = rand_uniform(
        N_boundary_each,
        0.0,
        T
    )

    S_bmax = torch.full(
        (N_boundary_each, 1),
        S_max,
        device=device,
        dtype=dtype
    )

    u_bmax_target = torch.zeros_like(t_bmax)

    return {
        "t_f": t_f,
        "S_f": S_f,
        "t_T": t_T,
        "S_T": S_T,
        "u_T_target": u_T_target,
        "t_b0": t_b0,
        "S_b0": S_b0,
        "u_b0_target": u_b0_target,
        "t_bmax": t_bmax,
        "S_bmax": S_bmax,
        "u_bmax_target": u_bmax_target,
    }

## 7. Loss function

In [ ]:
mse = nn.MSELoss()


def compute_losses(model, points):
    r = model.r()
    sigma = model.sigma()

    u_pred_data = model(t_data, S_data)

    squared_data_error = (u_pred_data - u_data) ** 2
    loss_data = torch.sum(
        data_weights * squared_data_error
    ) / torch.sum(data_weights)

    if torch.any(informative_continuation):
        loss_data_cont = torch.mean(
            squared_data_error[informative_continuation]
        )
    else:
        loss_data_cont = torch.tensor(
            float("nan"),
            device=device,
            dtype=dtype
        )

    noninformative_mask = ~informative_continuation

    if torch.any(noninformative_mask):
        loss_data_other = torch.mean(
            squared_data_error[noninformative_mask]
        )
    else:
        loss_data_other = torch.tensor(
            float("nan"),
            device=device,
            dtype=dtype
        )

    F, u_f, _, _, _ = american_bs_operator(
        model,
        points["t_f"],
        points["S_f"],
        r,
        sigma,
        q=q,
        create_graph=True
    )

    payoff_u = torch.relu(
        torch.tensor(K, device=device, dtype=dtype)
        - points["S_f"]
    ) / K

    gap_u = u_f - payoff_u

    vi_res = torch.minimum(-F, gap_u)
    loss_vi = torch.mean(vi_res ** 2)

    loss_tv = torch.mean(
        torch.relu(-gap_u) ** 2
    )

    loss_bs = torch.mean(
        torch.relu(F) ** 2
    )

    loss_eq = torch.mean(
        (F * gap_u) ** 2
    )

    continuation_mask = (
        gap_u.detach() > continuation_eps_u
    ).squeeze(-1)

    if torch.any(continuation_mask):
        loss_pde_cont = torch.mean(
            F[continuation_mask] ** 2
        )
        continuation_collocation_fraction = (
            continuation_mask.to(dtype).mean()
        )
    else:
        loss_pde_cont = torch.zeros(
            (),
            device=device,
            dtype=dtype
        )
        continuation_collocation_fraction = torch.zeros(
            (),
            device=device,
            dtype=dtype
        )

    u_T_pred = model(
        points["t_T"],
        points["S_T"]
    )

    loss_terminal = mse(
        u_T_pred,
        points["u_T_target"]
    )

    u_b0_pred = model(
        points["t_b0"],
        points["S_b0"]
    )

    loss_boundary_left = mse(
        u_b0_pred,
        points["u_b0_target"]
    )

    if USE_RIGHT_BOUNDARY:
        u_bmax_pred = model(
            points["t_bmax"],
            points["S_bmax"]
        )
        loss_boundary_right = mse(
            u_bmax_pred,
            points["u_bmax_target"]
        )
    else:
        loss_boundary_right = torch.zeros(
            (),
            device=device,
            dtype=dtype
        )

    total = (
        lambda_data * loss_data
        + lambda_vi * loss_vi
        + lambda_tv * loss_tv
        + lambda_terminal * loss_terminal
        + lambda_boundary_left * loss_boundary_left
        + lambda_boundary_right * loss_boundary_right
    )

    return {
        "total": total,
        "data": loss_data,
        "data_cont": loss_data_cont,
        "data_other": loss_data_other,
        "vi": loss_vi,
        "tv": loss_tv,
        "bs": loss_bs,
        "eq": loss_eq,
        "pde_cont": loss_pde_cont,
        "cont_colloc_fraction": continuation_collocation_fraction,
        "terminal": loss_terminal,
        "boundary_left": loss_boundary_left,
        "boundary_right": loss_boundary_right,
        "r": r,
        "sigma": sigma,
    }

## 8. CRR American put pricer

CRR means Cox Ross Rubinstein. It is the binomial tree model used here to reprice the American put after the inverse PINN has recovered \(\hat r\) and \(\hat\sigma\).

For each observed \((t,S)\), the remaining maturity is

\[
	au = T-t.
\]

MSE2 uses the recovered parameters in the CRR tree and compares those CRR prices with the synthetic prices.

In [ ]:
def crr_american_put(S0, tau, K, r, sigma, q=0.0, N=4000):
    S0 = float(S0)
    tau = float(tau)

    if tau <= 1e-14:
        return max(K - S0, 0.0)

    if S0 <= 1e-14:
        return float(K)

    dt = tau / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    disc = math.exp(-r * dt)

    p = (
        math.exp((r - q) * dt) - d
    ) / (u - d)

    if p < 0.0 or p > 1.0:
        raise ValueError(
            f"Invalid CRR probability p={p:.6f}"
        )

    j = np.arange(N + 1, dtype=np.float64)

    stock = (
        S0
        * np.power(u, j)
        * np.power(d, N - j)
    )

    values = np.maximum(
        K - stock,
        0.0
    )

    for i in range(N - 1, -1, -1):
        continuation = disc * (
            p * values[1:i + 2]
            + (1.0 - p) * values[:i + 1]
        )

        j_i = np.arange(
            i + 1,
            dtype=np.float64
        )

        stock_i = (
            S0
            * np.power(u, j_i)
            * np.power(d, i - j_i)
        )

        exercise = np.maximum(
            K - stock_i,
            0.0
        )

        values = np.maximum(
            continuation,
            exercise
        )

    return float(values[0])


def crr_prices_for_observations(r, sigma):
    prices = np.empty(
        len(american_df),
        dtype=np.float64
    )

    start = time.time()

    for i, row in american_df.iterrows():
        tau = T - float(row["t"])

        prices[i] = crr_american_put(
            S0=float(row["S"]),
            tau=tau,
            K=K,
            r=r,
            sigma=sigma,
            q=q,
            N=CRR_STEPS
        )

        if (i + 1) % 100 == 0:
            elapsed = time.time() - start
            print(
                f"CRR priced {i + 1}/{len(american_df)} "
                f"observations in {elapsed/60:.2f} minutes"
            )

    return prices

## 9. Train one random run

In [ ]:
def train_one_run(run_id):
    print()
    print("=" * 80)
    print(f"Run {run_id}/{N_RUNS}")
    print("=" * 80)

    model = AmericanInversePINN(
        r_initial=r_initial,
        sigma_initial=sigma_initial
    ).to(
        device=device,
        dtype=dtype
    )

    points = sample_run_points()

    optimizer = torch.optim.Adam([
        {
            "params": model.net.parameters(),
            "lr": lr_nn,
        },
        {
            "params": [model.raw_r],
            "lr": lr_r,
        },
        {
            "params": [model.raw_sigma],
            "lr": lr_sigma,
        },
    ])

    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        losses = compute_losses(
            model,
            points
        )

        losses["total"].backward()
        optimizer.step()

        if epoch == 1 or epoch % print_every == 0:
            row = {
                "run": run_id,
                "epoch": epoch,
                **{
                    k: float(v.detach().cpu())
                    for k, v in losses.items()
                }
            }

            history.append(row)

            print(
                f"Run {run_id:2d} | "
                f"Epoch {epoch:6d} | "
                f"total={row['total']:.3e} | "
                f"data={row['data']:.3e} | "
                f"vi={row['vi']:.3e} | "
                f"T={row['terminal']:.3e} | "
                f"B0={row['boundary_left']:.3e} | "
                f"Bmax={row['boundary_right']:.3e} | "
                f"r={row['r']:.6f} | "
                f"sigma={row['sigma']:.6f}"
            )

    adam_elapsed = time.time() - start

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"training_history_run{run_id}.csv"
        ),
        index=False
    )

    if RUN_LBFGS:
        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=1.0,
            max_iter=lbfgs_max_iter,
            history_size=50,
            tolerance_grad=1e-12,
            tolerance_change=1e-14,
            line_search_fn="strong_wolfe",
        )

        def closure():
            lbfgs.zero_grad()
            loss = compute_losses(
                model,
                points
            )["total"]
            loss.backward()
            return loss

        lbfgs.step(closure)

    r_hat = model.r().detach().cpu().item()
    sigma_hat = model.sigma().detach().cpu().item()

    model.eval()

    with torch.no_grad():
        u_pinn = model(
            t_data,
            S_data
        ).cpu().numpy().ravel()

    V_pinn = K * u_pinn

    mse1_pinn = np.mean(
        (V_pinn - V_data_np) ** 2
    )

    print(
        f"Run {run_id} recovered "
        f"r={r_hat:.8f}, sigma={sigma_hat:.8f}"
    )

    print(
        f"Run {run_id} MSE1 PINN={mse1_pinn:.10e}"
    )

    print(
        f"Run {run_id} starting CRR repricing for MSE2"
    )

    V_crr = crr_prices_for_observations(
        r=r_hat,
        sigma=sigma_hat
    )

    mse2_crr = np.mean(
        (V_crr - V_data_np) ** 2
    )

    print(
        f"Run {run_id} MSE2 CRR={mse2_crr:.10e}"
    )

    fit_df = american_df.copy()
    fit_df["V_PINN"] = V_pinn
    fit_df["V_CRR_recovered"] = V_crr
    fit_df["PINN_error"] = V_pinn - V_data_np
    fit_df["CRR_error"] = V_crr - V_data_np

    fit_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"price_fit_run{run_id}.csv"
        ),
        index=False
    )

    torch.save(
        {
            "run": run_id,
            "model_state_dict": model.state_dict(),
            "r_hat": r_hat,
            "sigma_hat": sigma_hat,
            "r_true": r_true,
            "sigma_true": sigma_true,
            "r_initial": r_initial,
            "sigma_initial": sigma_initial,
            "S_min": S_min,
            "S_max": S_max,
            "hidden_width": hidden_width,
            "hidden_layers": hidden_layers,
            "epochs": epochs,
            "lr_nn": lr_nn,
            "lr_r": lr_r,
            "lr_sigma": lr_sigma,
            "MSE1_PINN_V": mse1_pinn,
            "MSE2_CRR_V": mse2_crr,
        },
        os.path.join(
            OUTPUT_DIR,
            f"american_inverse_checkpoint_run{run_id}.pt"
        )
    )

    result = {
        "run": run_id,
        "r_hat": r_hat,
        "sigma_hat": sigma_hat,
        "r_abs_error": abs(r_hat - r_true),
        "sigma_abs_error": abs(sigma_hat - sigma_true),
        "MSE1_PINN_V": mse1_pinn,
        "MSE2_CRR_V": mse2_crr,
        "adam_minutes": adam_elapsed / 60.0,
    }

    del model
    del points
    del optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, history_df

## 10. Run the experiment 10 times

In [ ]:
run_results = []
all_histories = []

for run_id in range(1, N_RUNS + 1):
    result, history_df = train_one_run(
        run_id=run_id
    )

    run_results.append(result)
    all_histories.append(history_df)

    pd.DataFrame(
        run_results
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "run_results_in_progress.csv"
        ),
        index=False
    )

run_results_df = pd.DataFrame(
    run_results
)

all_history_df = pd.concat(
    all_histories,
    ignore_index=True
)

run_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "run_results.csv"
    ),
    index=False
)

all_history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "all_training_histories.csv"
    ),
    index=False
)

display(run_results_df)

## 11. Ten run averages

In [ ]:
summary_df = pd.DataFrame([{
    "n_runs": len(run_results_df),
    "r_hat_mean": run_results_df["r_hat"].mean(),
    "r_hat_std": run_results_df["r_hat"].std(ddof=1),
    "sigma_hat_mean": run_results_df["sigma_hat"].mean(),
    "sigma_hat_std": run_results_df["sigma_hat"].std(ddof=1),
    "MSE1_PINN_V_mean": run_results_df["MSE1_PINN_V"].mean(),
    "MSE1_PINN_V_std": run_results_df["MSE1_PINN_V"].std(ddof=1),
    "MSE2_CRR_V_mean": run_results_df["MSE2_CRR_V"].mean(),
    "MSE2_CRR_V_std": run_results_df["MSE2_CRR_V"].std(ddof=1),
    "r_true": r_true,
    "sigma_true": sigma_true,
}])

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "10_run_summary.csv"
    ),
    index=False
)

display(summary_df.T)

print()
print(
    "Average MSE1 PINN =",
    summary_df.loc[0, "MSE1_PINN_V_mean"]
)

print(
    "Average MSE2 CRR =",
    summary_df.loc[0, "MSE2_CRR_V_mean"]
)

print(
    "Average recovered r =",
    summary_df.loc[0, "r_hat_mean"]
)

print(
    "Average recovered sigma =",
    summary_df.loc[0, "sigma_hat_mean"]
)

## 12. Run to run diagnostic plots

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    run_results_df["run"],
    run_results_df["r_hat"],
    marker="o",
    label="Recovered r"
)
plt.axhline(
    r_true,
    linestyle="--",
    label="True r"
)
plt.xlabel("Run")
plt.ylabel("r")
plt.title("Recovered Interest Rate Across 10 Runs")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    run_results_df["run"],
    run_results_df["sigma_hat"],
    marker="o",
    label="Recovered sigma"
)
plt.axhline(
    sigma_true,
    linestyle="--",
    label="True sigma"
)
plt.xlabel("Run")
plt.ylabel("sigma")
plt.title("Recovered Volatility Across 10 Runs")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.semilogy(
    run_results_df["run"],
    run_results_df["MSE1_PINN_V"],
    marker="o",
    label="MSE1 PINN"
)
plt.semilogy(
    run_results_df["run"],
    run_results_df["MSE2_CRR_V"],
    marker="o",
    label="MSE2 CRR"
)
plt.xlabel("Run")
plt.ylabel("MSE")
plt.title("Price MSE Across 10 Runs")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Average training trajectories

In [ ]:
trajectory_mean = (
    all_history_df
    .groupby("epoch", as_index=False)
    .agg(
        r_mean=("r", "mean"),
        r_std=("r", "std"),
        sigma_mean=("sigma", "mean"),
        sigma_std=("sigma", "std"),
        total_mean=("total", "mean"),
        data_mean=("data", "mean"),
        vi_mean=("vi", "mean"),
    )
)

trajectory_mean.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "average_training_trajectory.csv"
    ),
    index=False
)

plt.figure(figsize=(8, 5))
plt.plot(
    trajectory_mean["epoch"],
    trajectory_mean["r_mean"],
    label="Mean recovered r"
)
plt.axhline(
    r_true,
    linestyle="--",
    label="True r"
)
plt.xlabel("Epoch")
plt.ylabel("r")
plt.title("Average Interest Rate Trajectory")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    trajectory_mean["epoch"],
    trajectory_mean["sigma_mean"],
    label="Mean recovered sigma"
)
plt.axhline(
    sigma_true,
    linestyle="--",
    label="True sigma"
)
plt.xlabel("Epoch")
plt.ylabel("sigma")
plt.title("Average Volatility Trajectory")
plt.legend()
plt.tight_layout()
plt.show()